In [1]:
pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import string

from nltk.corpus import stopwords
import nltk

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
nltk.download("stopwords")

In [ ]:
df = pd.read_csv(
    "IMDB Dataset.csv"
)

df.head()

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

df.head()

In [ ]:
df["review"] = df["review"].astype(str).str.lower()

df[["review", "sentiment"]].head()

In [ ]:
def remove_html(text):

    clean = re.compile("<.*?>")

    return re.sub(clean, " ", text)


df["clean_review"] = df["review"].apply(remove_html)

df[["review", "clean_review"]].head()

In [ ]:
def remove_special_characters(text):

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_review"] = df["clean_review"].apply(
    remove_special_characters
)

df[["clean_review", "sentiment"]].head()

In [ ]:
stop_words = set(stopwords.words("english"))
negation_words = {
    "no",
    "not",
    "never",
    "neither",
    "nor"
}
stop_words_without_negation = (
    stop_words - negation_words
)
def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word
        for word in words
        if word not in stop_words_without_negation
    ]
    return " ".join(filtered_words)

In [ ]:
print("Original Review:\n")

print(df["review"].iloc[0])

print("\n" + "=" * 80 + "\n")

print("Cleaned Review:\n")

print(df["clean_review"].iloc[0])

In [ ]:
# Encode Labels
df["label"] = df["sentiment"].map(
    {
        "negative": 0,
        "positive": 1
    }
)

df[["sentiment", "label"]].head()

In [ ]:
print(df["label"].value_counts())

In [ ]:
X = df["clean_review"]

y = df["label"]


X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)


X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

In [ ]:
print("Training Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Test Data:", X_test.shape)

In [ ]:
VOCAB_SIZE = 20000


tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)


tokenizer.fit_on_texts(X_train)

In [ ]:
print(
    "Number of Words in Vocabulary:",
    len(tokenizer.word_index)
)

In [ ]:
X_train_sequences = tokenizer.texts_to_sequences(
    X_train
)

X_val_sequences = tokenizer.texts_to_sequences(
    X_val
)

X_test_sequences = tokenizer.texts_to_sequences(
    X_test
)

In [ ]:
print("Original Review:")

print(X_train.iloc[0])

print("\nTokenized Sequence:")

print(X_train_sequences[0])

In [ ]:
MAX_LENGTH = 200


X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)


X_val_padded = pad_sequences(
    X_val_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)


X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [ ]:
print("Training Shape:", X_train_padded.shape)

print("Validation Shape:", X_val_padded.shape)

print("Test Shape:", X_test_padded.shape)

In [ ]:
import pickle
import os

os.makedirs("processed_data", exist_ok=True)

In [ ]:
with open(
    "processed_data/tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(tokenizer, file)

In [ ]:
np.save(
    "processed_data/X_train.npy",
    X_train_padded
)

np.save(
    "processed_data/X_val.npy",
    X_val_padded
)

np.save(
    "processed_data/X_test.npy",
    X_test_padded
)


np.save(
    "processed_data/y_train.npy",
    y_train
)

np.save(
    "processed_data/y_val.npy",
    y_val
)

np.save(
    "processed_data/y_test.npy",
    y_test
)

In [ ]:
print("Phase 2 preprocessing completed successfully!")

print("\nFinal Data Shapes:")

print("X_train:", X_train_padded.shape)
print("X_val:", X_val_padded.shape)
print("X_test:", X_test_padded.shape)

print("\nVocabulary Size:", VOCAB_SIZE)
print("Maximum Sequence Length:", MAX_LENGTH)